In [1]:
import cv2
import logging
import math
import numpy as np
import os
import pandas as pd
import pickle
import time

from pathlib import Path
from classes.preprocessing import preprocessing
from classes.video_handler import video_handler
#from classes.video_processing import video_processing
#from classes.image_processing import image_processing
from classes.event_extraction import event_extraction
from classes.pupil_prediction import pupil_prediction
from classes.interactive_plot import interactive_plot

In [2]:
initialDir = os.getcwd()

In [4]:
os.chdir(initialDir)

In [6]:
dataDir = initialDir + '\\Example' # change it to the head directiory of the data folder
inputDir = dataDir + '\\Input' # change it to the head directiory of the data folder
outputDir = dataDir + "\\Output"

movieDir = inputDir + '\\Movies'
eyetrackingDir = inputDir + '\\Eyetracking' 
folder_modeling_result = "Modeling result"
folder_csv_result = "csv results"
# change to initial directory
os.chdir(initialDir)

#################Set up information for the eyetracking data and movie######################
### maximum luminance of the screen (luminance when white is showed on screen)
maxlum = 212

### What is the resolution of eyetracking data (also is the screen resolution)
eyetracking_height = 1080
eyetracking_width = 1920
eyetracking_samplingrate = 300
### What is the video (showed on the screen) resolution (respective to eyetracking resolution).
# *Note that it is not the resolution in the video file.* For example, if the resolution of the eye-tracking data is 1000x500 and the physical height and width of the video displayed is half of the physical height and width of the screen, then videoRealHeight & videoRealHeight should be 500 and 250
  
videoRealHeight = 1080
videoRealWidth = 1920
# what is the physical width of the screen? (in cm)
screen_width = 59.5

# what is the distance between the eye and the monitor? (in cm)
eye_to_screen = 65

# What should be the size of the regional weight map? (relative to the size of the video) 
# Default value is the same size of the video horizontal visual angle. If the video is very large, consider make it smaller
degVF_param = 1

## if video resolution is not the same as eyetracking resolution, what color is the screen covered with? (enter rgb value, E.g. r,b,g = 0 means black)
screenBgColorR = 0
screenBgColorG = 0
screenBgColorB = 0

# select parameters for event extraction and modeling
# event extraction mode (default: gaze-centered)
gazecentered = True
# map type: can choose between square and circular (default: circular)
map_type = "circular"
# But if not gazecentered, map type can only be square 
if not gazecentered:
    map_type = "square"
# number of weight: (default: 44)
# if mapType is sqaure, can choose among 2 (left or right), 6 (original open-DPSM paper) and 48 (all regions separately)
# if mapType is circluar, can choose between 2 (left or right), and 44 (all regions separately)
n_weight = 44
# modeling type: model each movie separately or model all the movies together for each subject
modelingType = "separate" # all or separate
# whether to zscore the pupil data before modeling
pupil_zscore = False # default is false for this version. Modeling for the absolute pupil size, not just pupil size changes
# load other parameters that should not be changed
exec(open("settings.py").read())

### Do you want to save:
# - model evaluation & paramters
saveParams = True
# - data used for modeling
saveData = True

########################################################################################################################################################
###########################################################End of information entering.#################################################################
########################################################################################################################################################

#%%#########################################Preprocessing and visual even extraction####################################################################
# This is the indicator that the app is not used.
useApp = False
overall_lum_extracted = False
event_extracted = False
movieFiles = [file for file in os.listdir(movieDir)]
movieListAll = [file.split('.')[0] for file in movieFiles]
print(f"Selected parameters: \n- gazecentered: {gazecentered}\n- map type: {mapType}\n- number of weight: {nWeight}")

# read video data and check information
prepObj = preprocessing()
if "eyetrackingDir" not in globals():
    print("No eyetracking data! Please check the data.")
else:
    if gazecentered:
        print("Eyetracking folder found.")
        subjects = os.listdir(eyetrackingDir)
    else:
        # all the subjects will have the same pickle files
        subjects = ['sc'] 
# iteratively extract events for all the subjects and all the movies

Selected parameters: 
- gazecentered: True
- map type: circular
- number of weight: 44
Eyetracking folder found.


In [7]:
movieFiles

['1-2.mp4', '1-3.mp4', '2-1.mp4', '2-2.mp4', '3-1.mp4', '3-2.mp4']

In [8]:
subjects

['s1', 's2']

In [28]:
import importlib
import classes.event_extraction
importlib.reload(classes.event_extraction)
from classes.event_extraction import event_extraction
# iteratively extract events for all the subjects and all the movies
for subjectName in subjects:
    if gazecentered:
        # If gazecentered, check the files from each subject folder
        subjectDir = eyetrackingDir + f"\\{subjectName}"
        ### one folder for one participant, containing all eyetracking data 
        csvFiles = [file for file in os.listdir(subjectDir) if file.endswith('.csv')]
    
        #check whether one eyetracking data is paired with one movie file
        noMovieFiles = [file for file in csvFiles if file.replace(".csv",'') not in movieListAll]
        if len(noMovieFiles)>0:
            print(f"Warning: {len(noMovieFiles)} movies not found")
        movieList = [file.replace('.csv','') for file in csvFiles]
    else:
        # If not gazecentered, check the files from Movies folder (eyetracking data does not matter)
        movieList = [file.split(".")[0] for file in os.listdir(movieDir)]
    for movie in movieList:
        # check video information
        movieName = [file for file in os.listdir(movieDir) if file.startswith(movie)][0]
        filename_movie = movieDir +"\\" + movieName 
        prepObj.videoFileName = filename_movie
        #prepObj.preprocessingVideo()
        prepObj.loadVideo(filename_movie)
        prepObj.getVideoInfo()
        # extract information from video file
        video_nFrame = prepObj.vidInfo['nFrames']
        video_height = prepObj.vidInfo['height']
        video_width = prepObj.vidInfo['width']
        video_ratio = video_height / video_width 
        video_duration = prepObj.vidInfo['duration']
        video_fps = prepObj.vidInfo['fps']
        movieName = movieName.split(".")[0]
        # read eyetracking data if gaze centered
        if gazecentered:
            filename_csv = subjectDir + "\\" + movieName +".csv"
            # read eyetracking data and check information
            df_eyetracking = pd.read_csv(filename_csv, index_col=None, header = None,sep = ",") # Change it to sep = ',' if encounter error
            # change the beginning as 0s
            eyetracking_duration = df_eyetracking.iloc[-1,0]
            eyetracking_nSample = df_eyetracking.shape[0]
            df_eyetracking.iloc[:,0] = df_eyetracking.iloc[:,0]-df_eyetracking.iloc[0,0]
            if eyetracking_samplingrate != int(1/(eyetracking_duration/eyetracking_nSample)):
                df_eyetracking.iloc[:,0] = df_eyetracking.iloc[:,0]/1000
                eyetracking_duration = df_eyetracking.iloc[-1,0]
    
        # check if video and the eyetracking data have the same ratio
        if videoRealHeight == eyetracking_height and videoRealWidth ==eyetracking_width:
            videoScreenSameRatio = True 
            videoStretched = True
        elif videoRealHeight == eyetracking_height or videoRealWidth ==eyetracking_width:
            videoScreenSameRatio = False
            videoStretched = True
        else:
            videoScreenSameRatio = False
            videoStretched = False
        #############################################Visual events extraction##############################################
        # calculate visual angle of the movie displayed 
        videoWidthCM = videoRealWidth / (eyetracking_width/screen_width)
        videoWidthDeg =math.degrees(math.atan(videoWidthCM/2/eye_to_screen))*2
        # visual angle of the regional weight map 
        degVF = videoWidthDeg *degVF_param
        # create folder to save data
        os.chdir(dataDir) 
        foldername = "Output"
        if not os.path.exists(foldername):
           os.makedirs(foldername)
        os.chdir(foldername)
        foldername = "Visual events"
        if not os.path.exists(foldername):
           os.makedirs(foldername)
        os.chdir(foldername)
        # name the feature extracted pickle:
        if mapType == "square":
            picklename ="square_" + movieName + "_"+ subjectName + "_VF_" +colorSpace + "_nWeight_" + str(nWeight)  + ".pickle"
        elif mapType == "circular":
            picklename ="circular_" + movieName + "_"+ subjectName + "_VF_" +colorSpace + "_nWeight_" + str(nWeight) + ".pickle"
        overall_lum_name = "overall_lum.csv"
        if picklename in os.listdir():
            # skip event extraction if it has already done previously
            print(f"Subject {subjectName} Movie {movieName} already extracted")

            event_extracted = True
        if overall_lum_name in os.listdir():
            df_overall_lum = pd.read_csv(overall_lum_name)
            if df_overall_lum.loc[df_overall_lum['movie'] == movie, 'overall_lum'].values[0] >0:
                overall_lum_extracted = True
            else:
                print(f"Overall luminance of movie {movieName} not extracted")
                overall_lum_extracted = False
        else:
            overall_lum_extracted = False
            df_overall_lum = pd.DataFrame(columns = ['movie', 'overall_lum'])
            df_overall_lum['movie'] = movieList
            print(f"Overall luminance of movie {movieName} not extracted")
        overall_lum_extracted = False
        event_extracted = False
        if not event_extracted or not overall_lum_extracted:
            # start of event extraction for one movie in one subject
            if not event_extracted:
                print(f"====Extracting events for subject {subjectName} movie {movieName}====")
            if not overall_lum_extracted:
                print(f"====Extracting overall luminance for movie {movieName}====")
            print(f"Video number of frame: {video_nFrame}")
            print(f"Video height x width: {video_height}x{video_width}; aspect ratio (width:height): {1/video_ratio}")
            print(f"Video duration: {video_duration}")
            print(f"Video frame rate: {video_fps}")
            if gazecentered:
                print(f"Eyetracking data duration: {eyetracking_duration} seconds")
                print(f"Eyetracking data sampling rate: {eyetracking_samplingrate} Hz")
            # feature extraction class
            eeObj = event_extraction()
            # load some data and parameters
            eeObj.video_duration = video_duration
            eeObj.video_fps = video_fps
            eeObj.stretchToMatch = stretchToMatch
            eeObj.subject = subjectName
            eeObj.movieNum = movieName
            eeObj.picklename = picklename
            eeObj.filename_movie = filename_movie
            eeObj.setNBufFrames(nFramesSeqImageDiff + 1)
            eeObj.imCompFeatures = True  # creates: imageObj.vectorMagnFrame
            eeObj.showVideoFrames = showVideoFrames
            eeObj.imColSpaceConv = colorSpace
            eeObj.gazecentered = gazecentered
            eeObj.nVertMatPartsPerLevel = nVertMatPartsPerLevel  # [4, 8, 16, 32]
            eeObj.aspectRatio = aspectRatio 
            eeObj.imageSector = imageSector
            eeObj.nFramesSeqImageDiff = nFramesSeqImageDiff
            eeObj.selectFeatures = featuresOfInterest
            eeObj.scrGamFac = scrGamFac
            
            eeObj.maxlum = maxlum
            eeObj.useApp = useApp
               
            eeObj.vidInfo = prepObj.vidInfo # extract vidInfo from preprocessing object
            eeObj.mapType = mapType
            eeObj.degVF = degVF
            eeObj.eye_to_screen = eye_to_screen
            eeObj.screen_width= screen_width
            eeObj.nWeight = nWeight
            # process eyetracking data
            if gazecentered: # if there is eyetracking data, do gaze-contingent visual events extraction
                eeObj.eyetracking_duration = eyetracking_duration
                eeObj.eyetracking_height = eyetracking_height
                eeObj.eyetracking_width = eyetracking_width
                eeObj.eyetracking_samplingrate = eyetracking_samplingrate
                eeObj.videoRealHeight = videoRealHeight
                eeObj.videoRealWidth = videoRealWidth
                eeObj.screenBgColorR = screenBgColorR
                eeObj.screenBgColorG = screenBgColorG
                eeObj.screenBgColorB = screenBgColorB
                eeObj.videoScreenSameRatio = videoScreenSameRatio 
                eeObj.videoStretched = videoStretched 
                timeStampsSec = np.array(df_eyetracking.iloc[:,0])
                gazexdata = np.array(df_eyetracking.iloc[:,1])
                gazeydata = np.array(df_eyetracking.iloc[:,2])
                pupildata = np.array(df_eyetracking.iloc[:,3])
                # resample the eytracking data to match the video sampling rate
                eeObj.sampledTimeStamps_featureExtraction =eeObj.prepare_sampleData(timeStampsSec, video_nFrame)
                eeObj.sampledgazexData_featureExtraction = eeObj.prepare_sampleData(gazexdata, video_nFrame)
                eeObj.sampledgazeyData_featureExtraction = eeObj.prepare_sampleData(gazeydata, video_nFrame)
                eeObj.sampledpupilData_featureExtraction = eeObj.prepare_sampleData(pupildata, video_nFrame)
            # Event extraction function: 
            # This can take a while. The extracted features will be saved in folder "Visual events"
            if not event_extracted:
                eeObj.run()

Subject s1 Movie 1-2 already extracted
====Extracting events for subject s1 movie 1-2====
====Extracting overall luminance for movie 1-2====
Video number of frame: 7500
Video height x width: 1080x1920; aspect ratio (width:height): 1.7777777777777777
Video duration: 300.0
Video frame rate: 25
Eyetracking data duration: 0.29999581999999997 seconds
Eyetracking data sampling rate: 300 Hz
59.5 32.26890756302521 1920


AttributeError: 'NoneType' object has no attribute 'reshape'

In [29]:
wedges_per_annulus = [4, 8, 8, 12, 12]
for a,b in enumerate(wedges_per_annulus ):
    print(a,b)

0 4
1 8
2 8
3 12
4 12
